# Bitcoin Price Predictor

- This project is designed to predict the price of Bitcoin based on two datasets
  - https://huggingface.co/datasets/ckandemir/bitcoin_tweets_sentiment_kaggle
  - https://huggingface.co/datasets/WinkingFace/CryptoLM-Bitcoin-BTC-USDT


In [84]:
#imports
from datasets import load_dataset
from datetime import datetime
from pydantic import BaseModel
from collections import defaultdict, Counter
from openai import OpenAI
import random
import json
import gradio as gr


In [2]:
START_DATE = "2019-06-26"
END_DATE = "2021-06-26"

In [9]:

# fetch datasets
bitcoin_tweets = load_dataset("ckandemir/bitcoin_tweets_sentiment_kaggle")
bitcoin_price = load_dataset("WinkingFace/CryptoLM-Bitcoin-BTC-USDT", split="train")


Resolving data files:   0%|          | 0/92 [00:00<?, ?it/s]

In [10]:
def merge_splits(dataset_dict):
    merged = []
    for split in dataset_dict.keys(): 
        merged.extend(dataset_dict[split])
    return merged

In [ ]:
def parse_prices(dataset):
    sorted_dataset = dataset.sort("timestamp")
    
    daily_last_price = {}
    
    for datapoint in sorted_dataset:
        date_only = str(datapoint["timestamp"].date())
        daily_last_price[date_only] = datapoint["close"]
    
    results = [ {"date": date, "price": price} for date, price in daily_last_price.items()]
    
    return results



In [12]:
prices_data = parse_prices(bitcoin_price)

In [61]:
def parse_tweets(dataset):
    grouped = defaultdict(list)

    for datapoint in dataset:
        date_obj = datetime.strptime(datapoint["Date"], "%Y-%m-%d")
        date_str = f"{date_obj.year}-{date_obj.month}-{date_obj.day}"  # "YYYY-M-D"
        grouped[date_str].append(datapoint["text"])

    result = []
    for date_str in sorted(grouped.keys(), key=lambda x: datetime.strptime(x, "%Y-%m-%d")):
        tweets = grouped[date_str]

        combined_tweets = "\n".join(f"{i+1} - {tweet}" for i, tweet in enumerate(tweets))

        result.append({
            "date": date_str,
            "tweet": combined_tweets
        })

    return result

In [62]:
all_tweets = merge_splits(bitcoin_tweets)

tweets_data = parse_tweets(all_tweets)

In [63]:
class Item(BaseModel):
    date: str
    tweet: str
    price: float

In [64]:
full_data = []

print("prices start date", prices_data[0]["date"]) 
print("prices end date", prices_data[-1]["date"])
print("tweets start date", tweets_data[0]["date"])
print("tweets end date", tweets_data[-1]["date"])
# will only use data from prices start date to tweets end date and dates with no tweets shouldn't be included

price_by_date = {p["date"]: p["price"] for p in prices_data}

start_date = prices_data[0]["date"]
end_date = tweets_data[-1]["date"]

for tweet in tweets_data:
    tweet_date = tweet["date"]
    
    # Only include tweets within the price date range
    if start_date <= tweet_date <= end_date and tweet_date in price_by_date:
        full_data.append(
            Item(
                date=tweet_date,
                tweet=tweet["tweet"],
                price=price_by_date[tweet_date]
            )
        )

print(f"Collected {len(full_data)} items")


prices start date 2017-08-18
prices end date 2025-03-19
tweets start date 2014-9-18
tweets end date 2019-7-14
Collected 130 items


In [72]:
random.seed(42)
random.shuffle(full_data)

In [73]:
def create_train_test_split(data, test_size=0.2):
    # Calculate the number of items for the test set
    test_size = int(len(data) * test_size)
    
    # Split the data into train and test sets
    train_data = data[:-test_size]
    test_data = data[-test_size:]

    return train_data, test_data

train_data, test_data = create_train_test_split(full_data)

print(f"Train data size: {len(train_data)}")
print(f"Test data size: {len(test_data)}")


Train data size: 104
Test data size: 26


In [74]:
def convert_to_jsonl_for_model(data, filename):
    with open(filename, "w") as f:
        for item in data:
            messages = {
                "messages": [
                    {
                        "role": "user",
                        "content": f"Estimate the price of bitcoin based on their tweets. Respond with the price, no explanation\n\n{item.tweet}"
                    },
                    {
                        "role": "assistant",
                        "content": f"{item.price}"
                    }
                ]
            }
            f.write(json.dumps(messages) + "\n")

In [75]:
convert_to_jsonl_for_model(train_data, "train_data.jsonl")
convert_to_jsonl_for_model(test_data, "test_data.jsonl")

In [79]:
openai = OpenAI()

In [80]:
# create files in openai
with open("train_data.jsonl", "rb") as f:
    train_file = openai.files.create(
        file=f,
        purpose="fine-tune"
    )

train_file


FileObject(id='file-8faB4LoTA4qF79beFo3JWZ', bytes=505601, created_at=1772210881, filename='train_data.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)

In [81]:
# create files in openai
with open("test_data.jsonl", "rb") as f:
    test_file = openai.files.create(
        file=f,
        purpose="fine-tune"
    )

test_file


FileObject(id='file-HnfjYUKt8M3TMeNeQBPGZe', bytes=50820, created_at=1772210891, filename='test_data.jsonl', object='file', purpose='fine-tune', status='processed', expires_at=None, status_details=None)

In [82]:
openai.fine_tuning.jobs.create(
    training_file=train_file.id,
    validation_file=test_file.id,
    model="gpt-4.1-nano-2025-04-14",
    seed=42,
    hyperparameters={"n_epochs": 1, "batch_size": 1},
    suffix="bitcoin_predictore"
)

FineTuningJob(id='ftjob-tQmRJK9dhuylmtsGzSMpmQru', created_at=1772210956, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size=1, learning_rate_multiplier='auto', n_epochs=1), model='gpt-4.1-nano-2025-04-14', object='fine_tuning.job', organization_id='org-RdJj9hNtM0zKXMeQiGs0WYmY', result_files=[], seed=42, status='validating_files', trained_tokens=None, training_file='file-8faB4LoTA4qF79beFo3JWZ', validation_file='file-HnfjYUKt8M3TMeNeQBPGZe', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=1, learning_rate_multiplier='auto', n_epochs=1))), user_provided_suffix='bitcoin_predictore', usage_metrics=None, shared_with_openai=False, eval_id=None, internal_worker_backend=None)

In [83]:
openai.fine_tuning.jobs.list(limit=1)

SyncCursorPage[FineTuningJob](data=[FineTuningJob(id='ftjob-tQmRJK9dhuylmtsGzSMpmQru', created_at=1772210956, error=Error(code=None, message=None, param=None), fine_tuned_model=None, finished_at=None, hyperparameters=Hyperparameters(batch_size=1, learning_rate_multiplier='auto', n_epochs=1), model='gpt-4.1-nano-2025-04-14', object='fine_tuning.job', organization_id='org-RdJj9hNtM0zKXMeQiGs0WYmY', result_files=[], seed=42, status='validating_files', trained_tokens=None, training_file='file-8faB4LoTA4qF79beFo3JWZ', validation_file='file-HnfjYUKt8M3TMeNeQBPGZe', estimated_finish=None, integrations=[], metadata=None, method=Method(type='supervised', dpo=None, reinforcement=None, supervised=SupervisedMethod(hyperparameters=SupervisedHyperparameters(batch_size=1, learning_rate_multiplier='auto', n_epochs=1))), user_provided_suffix='bitcoin_predictore', usage_metrics=None, shared_with_openai=False, eval_id=None, internal_worker_backend=None)], has_more=True, object='list')

In [94]:
job_id = openai.fine_tuning.jobs.list(limit=1).data[0].id
fine_tuned_model_name = openai.fine_tuning.jobs.retrieve(job_id).fine_tuned_model

In [95]:
system_prompt = """
You are a bitcoin price predictor. You are given a tweet and you need to predict the price of bitcoin based on the tweet.
You are friendly and helpful, give an estimate of the price of bitcoin based on the tweet.
"""
def predict(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(
        model=fine_tuned_model_name,
        messages=messages,
        max_tokens=100
    )
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(
    predict,
    title="Bitcoin Price Predictor",
    type="messages"
).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.
